### 105. Construct Binary Tree from Preorder and Inorder Traversal

preorder: 根節點、左節點、右節點  
inorder: 左節點、根節點、右節點

In [1]:
# Definition for a binary tree node.
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

### DFS (Optimal)

- 時間複雜度 $O(n)$
  - 樹中的每個節點都剛好被建立一次，且 preorder_index 與 inorder_index 最多各遞增 $n$ 次。
- 空間複雜度 $O(n)$
  - 此解法沒有使用任何雜湊表，其空間完全取決於遞迴時系統棧（Call Stack）的深度。在最壞情況下（樹呈現單向鏈狀），遞迴深度為 $n$，故為 $O(n)$。

炫技儲備

In [2]:
from typing import List, Optional

class Solution:
    def buildTree(self, preorder: List[int], inorder: List[int]) -> Optional[TreeNode]:
        # 紀錄目前處理到的前序遍歷和中序遍歷的索引位置
        self.preorder_index = 0
        self.inorder_index = 0

        def dfs(parent_node_limit):
            # 若前序索引已超出範圍，代表樹已構建完畢，返回 None
            if self.preorder_index >= len(preorder):
                return None

            # 如果目前中序節點值等於父節點限制值，代表左子樹已完成，返回 None
            if inorder[self.inorder_index] == parent_node_limit:
                self.inorder_index += 1  # 移動到下一個中序節點
                return None

            # 建立目前的根節點
            root = TreeNode(preorder[self.preorder_index])
            self.preorder_index += 1  # 移動到下一個前序節點

            # 遞迴構建左子樹，左子樹的邊界為當前 root 的值
            root.left = dfs(parent_node_limit = root.val)
            # 遞迴構建右子樹，右子樹的邊界與當前相同（父層傳下來的限制）
            root.right = dfs(parent_node_limit = parent_node_limit)

            return root

        # 初始呼叫，使用無窮大作為整棵樹的邊界
        return dfs(parent_node_limit = float("inf")) # time: O(n), space: O(n)

In [3]:
# Input: 
preorder = [3,9,20,15,7]
inorder  = [9,3,15,20,7]
# Output: [3,9,20,null,null,15,7]

Solution().buildTree(preorder, inorder)

📌 核心作用分點總結
- preorder 的作用（領路開路）：利用「根 $\rightarrow$ 左 $\rightarrow$ 右」的特性，負責提供新節點的數值。只要沒碰到邊界，preorder 的下一個元素永遠是當前要建立的子樹根節點。
- inorder 的作用（雷達剎車）：負責提供邊界限制。利用「左 $\rightarrow$ 根 $\rightarrow$ 右」的特性，當前指到的值就是目前允許向左/向右挖掘的最極限，碰到了就必須踩煞車回溯。
  
💡 想法驗證
1. 「以 preorder 的順序遍歷，透過 inorder 判斷左子樹末節點」
   - 以 preorder 的順序來遍歷，中間透過 inorder 的值來判斷是否已經到左子樹的末節點，因為 inorder 的第一個值一定是整棵樹最左下角的末節點。當 preorder 一路往左下角探底，探到數值與 inorder 當前的值一樣時，就代表「最左邊的邊界到了，不能再往左了」，此時左子樹遞迴踩煞車（返回 None）。
2. 「inorder 往後的根節點，是 preorder 從末節點往前推的根節點」
   - inorder 第一個值依序往後的根結點，就會是 preorder 從左子樹的末節點往前推（回溯）的根結點。當左下角撞到邊界返回後，程式會從末節點往回推給父節點。而在 inorder 中，左子樹走完緊接著就是「根」，所以當我們往回推時，執行 inorder_index += 1，這個雷達指針就會精準地跟著對上正在回溯的那個父節點，兩者達成同步。
3. 「右子樹同樣利用 inorder 的值來決定左下角終點」
   - 當我們轉向檢查右子樹時，右子樹自己也是一棵獨立的子樹，它會重複相同的邏輯——先一路往右子樹的「左下角」探底。此時，右子樹的左下角終點，就會由當初傳進來的 parent_node_limit（父層邊界）來卡住它。inorder 的指針在左子樹、根、右子樹依序被建立的過程中，從左到右一格一格被指針消費掉的。當整棵樹都建完時，inorder 就會剛好指到最後。
4. 「以 preorder 的 index 和長度比較來知道遍歷完」
   - 以 preorder 的 index 和 preorder 的長度比較來知道已經遍歷完所有節點了。因為 preorder 包含了所有的節點。當領路的指針把整個陣列走完，代表所有節點都變成樹的一部分了，這時不論 limit 是多少，一律直接返回 None 結束蓋樹工程。

### DFS + Hash Map

**時間複雜度: $O( n )$**  
**空間複雜度: $O( n )$**

Hash 對變形題最穩、最通用

In [4]:
from typing import List, Optional

class Solution:
    def buildTree(self, preorder: List[int], inorder: List[int]) -> Optional[TreeNode]:
        # 建立一個字典，用來快速查詢某個節點值在中序遍歷中的索引位置
        indices = {value: idx for idx, value in enumerate(inorder)} # time: O(n)

        # 使用類別屬性來紀錄目前在前序遍歷中的索引位置
        self.pre_index = 0

        def dfs(left, right):
            # 若左邊界超過右邊界，代表此區間無節點，回傳 None
            if left > right:
                return None
            
            root_value = preorder[self.pre_index] # 取得目前前序遍歷中的根節點值
            root = TreeNode(root_value) # 建立根節點
            mid = indices[root_value] # 取得此根節點在中序遍歷中的索引位置
            
            print("-" * 50)
            print(f"{indices = }")
            print(f"{preorder = }")
            print(f"{inorder = }")
            print(f"\n{left = }, {right = }")
            print(f"{preorder[left:right+1] = }")
            print(f"{inorder[left:right+1] = }")
            print(f"{self.pre_index = }, {root.val = }, {mid = }")

            self.pre_index += 1 # 將前序索引往右移，準備建構下一個節點

            # 遞迴建構左子樹，範圍為中序的 left 到 mid - 1
            root.left = dfs(left, mid - 1)

            # 遞迴建構右子樹，範圍為中序的 mid + 1 到 right
            root.right = dfs(mid + 1, right)
            print(f"{root.val = }")
            print(f"{root.left.val = }") if root.left else print(f"{root.left = }")
            print(f"{root.right.val =}") if root.right else print(f"{root.right = }")

            return root

        return dfs(0, len(preorder)-1) # time: O(n), space: O(n)

In [5]:
# Input: 
preorder = [3,9,20,15,7]
inorder  = [9,3,15,20,7]
# Output: [3,9,20,null,null,15,7]

Solution().buildTree(preorder, inorder)

--------------------------------------------------
indices = {9: 0, 3: 1, 15: 2, 20: 3, 7: 4}
preorder = [3, 9, 20, 15, 7]
inorder = [9, 3, 15, 20, 7]

left = 0, right = 4
preorder[left:right+1] = [3, 9, 20, 15, 7]
inorder[left:right+1] = [9, 3, 15, 20, 7]
self.pre_index = 0, root.val = 3, mid = 1
--------------------------------------------------
indices = {9: 0, 3: 1, 15: 2, 20: 3, 7: 4}
preorder = [3, 9, 20, 15, 7]
inorder = [9, 3, 15, 20, 7]

left = 0, right = 0
preorder[left:right+1] = [3]
inorder[left:right+1] = [9]
self.pre_index = 1, root.val = 9, mid = 0
root.val = 9
root.left = None
root.right = None
--------------------------------------------------
indices = {9: 0, 3: 1, 15: 2, 20: 3, 7: 4}
preorder = [3, 9, 20, 15, 7]
inorder = [9, 3, 15, 20, 7]

left = 2, right = 4
preorder[left:right+1] = [20, 15, 7]
inorder[left:right+1] = [15, 20, 7]
self.pre_index = 2, root.val = 20, mid = 3
--------------------------------------------------
indices = {9: 0, 3: 1, 15: 2, 20: 3, 7: 4}


### DFS

- 時間複雜度: $O( n^2 )$
  - 最壞情況下會遞迴 $n$ 層（深度），而每一層都要跑 $O(n)$ 的 .index() 與切片。
  - 兩者相乘：$n \times O(n) = O(n^2)$。
- 空間複雜度: $O( n^2 )$
  - 最壞情況下系統棧深度為 $n$。
  - 每層遞迴時，父層切出來的陣列還沒被釋放，子層又切出了新的陣列， $O( n )$

暴力遞迴解

In [6]:
from typing import List, Optional

class Solution:
    def buildTree(self, preorder: List[int], inorder: List[int]) -> Optional[TreeNode]: # time: O(n^2), space: O(n^2)
        # 如果前序或中序為空，代表沒有節點可建，返回 None
        if not preorder or not inorder:
            return None
        
        # 建立根節點，根節點為前序遍歷的第一個元素
        root = TreeNode(preorder[0])
        print(f"\n{preorder = }, {inorder = }")
        print(f"{root.val = }")

        # 在中序遍歷中找到根節點的位置，左邊的都是左子樹的節點
        mid = inorder.index(preorder[0]) # time: O(n)
        print(f"root的左邊有 ({mid = }) 個節點")

        preorder_left_nodes = preorder[1:mid+1] # preorder 中左子樹的部分，去除:root(第1個)，left的結尾會落在(mid+1)，總共才會是mid個
        preorder_right_nodes = preorder[mid+1:] # preorder 中右子樹的部分，去除:left(mid個)+root(1個)
        inorder_left_nodes = inorder[:mid] # inorder 中左子樹的部分
        inorder_right_nodes = inorder[mid+1:] # inorder 中右子樹的部分
        print(f"-> {preorder_left_nodes = }")
        print(f"-> {preorder_right_nodes = }")
        print(f"-> {inorder_left_nodes = }")
        print(f"-> {inorder_right_nodes = }")
        
        # 遞迴建立左子樹與右子樹
        root.left = self.buildTree(preorder=preorder_left_nodes, inorder=inorder_left_nodes) # space: O(n)
        root.right = self.buildTree(preorder=preorder_right_nodes, inorder=inorder_right_nodes) # space: O(n)
        print(f"{root.val = }")
        print(f"{root.left.val = }") if root.left else print(f"{root.left = }")
        print(f"{root.right.val =}") if root.right else print(f"{root.right = }")

        return root

In [7]:
# Input: 
preorder = [3,9,20,15,7]
inorder  = [9,3,15,20,7]
# Output: [3,9,20,null,null,15,7]

Solution().buildTree(preorder, inorder)


preorder = [3, 9, 20, 15, 7], inorder = [9, 3, 15, 20, 7]
root.val = 3
root的左邊有 (mid = 1) 個節點
-> preorder_left_nodes = [9]
-> preorder_right_nodes = [20, 15, 7]
-> inorder_left_nodes = [9]
-> inorder_right_nodes = [15, 20, 7]

preorder = [9], inorder = [9]
root.val = 9
root的左邊有 (mid = 0) 個節點
-> preorder_left_nodes = []
-> preorder_right_nodes = []
-> inorder_left_nodes = []
-> inorder_right_nodes = []
root.val = 9
root.left = None
root.right = None

preorder = [20, 15, 7], inorder = [15, 20, 7]
root.val = 20
root的左邊有 (mid = 1) 個節點
-> preorder_left_nodes = [15]
-> preorder_right_nodes = [7]
-> inorder_left_nodes = [15]
-> inorder_right_nodes = [7]

preorder = [15], inorder = [15]
root.val = 15
root的左邊有 (mid = 0) 個節點
-> preorder_left_nodes = []
-> preorder_right_nodes = []
-> inorder_left_nodes = []
-> inorder_right_nodes = []
root.val = 15
root.left = None
root.right = None

preorder = [7], inorder = [7]
root.val = 7
root的左邊有 (mid = 0) 個節點
-> preorder_left_nodes = []
-> preorder_right_